# Build `genes_df_subset` — CRISPRa gene models (primary assembly + alt/patch contigs)

Produces `genome/genes_df_subset_for_sgRNA_annotation.parquet`, the gene table (with per-gene TSS/CDS
lists) that Phase 5 uses for nearest-TSS gene assignment. Two parts:

1. **Primary assembly** — from the GENCODE v48 gene/transcript/CDS annotations (parsed parquets in the
   sibling GRNPerturbSeq project). Subset to protein-coding + IG/TR genes + every gene the CRISPRa
   library targets.
2. **Alt / patch / haplotype contigs** — some library genes (e.g. **GSTT1**, **LILRA3**) are annotated
   *only* on alternate scaffolds, so guides align to `*_alt` contigs that the primary annotation lacks.
   We parse `gencode.v48.chr_patch_hapl_scaff` for those contigs and remap GENCODE's GenBank contig names
   (`KI270879.1`) to the UCSC names the hg38 index uses (`chr22_KI270879v1_alt`) via `hg38.chromAlias.txt`.

Run under `gwt-env`. Inputs `gencode.v48.altcontigs.gtf` (pre-filtered to non-primary contigs) and
`hg38.chromAlias.txt` live in `genome/`.

In [1]:
import re
import numpy as np
import pandas as pd

GENOME = 'genome'
# Primary-assembly parsed GENCODE v48 annotations (sibling GRNPerturbSeq project)
REF_GENOME = ('/Users/rzhu/Gladstone Dropbox/Ronghui Zhu/GRNPerturbSeq/4_codes/'
              'GWT_perturbseq_analysis/src/5_sgRNA_annotation/genome')
master = pd.read_parquet(f'{GENOME.replace("genome","results")}/CRISPRa_targeting_sgRNA_master.parquet')
lib_ids = set(master['gene_id'].dropna())
print(f'library reconciled gene_ids: {len(lib_ids):,}')

library reconciled gene_ids: 18,942


In [2]:
# --- Acquire alt-contig inputs (idempotent: download/extract only if missing) ---
import os
import gzip
import urllib.request

# UCSC hg38 contig alias: maps GENCODE GenBank names (KI270879.1) <-> UCSC names (chr22_KI270879v1_alt)
alias_path = f'{GENOME}/hg38.chromAlias.txt'
if not os.path.exists(alias_path):
    print('downloading UCSC hg38 chromAlias...')
    urllib.request.urlretrieve(
        'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/latest/hg38.chromAlias.txt', alias_path)

# GENCODE v48 patch/hapl/scaff annotation -> keep only non-primary-contig gene/transcript/CDS records
alt_gz = f'{GENOME}/gencode.v48.altscaff.gtf.gz'
altcontigs = f'{GENOME}/gencode.v48.altcontigs.gtf'
if not os.path.exists(altcontigs):
    if not os.path.exists(alt_gz):
        print('downloading GENCODE v48 patch/hapl/scaff annotation (~60 MB)...')
        urllib.request.urlretrieve(
            'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/'
            'gencode.v48.chr_patch_hapl_scaff.annotation.gtf.gz', alt_gz)
    std = re.compile(r'^chr([0-9]+|X|Y|M)$')   # primary chromosomes to exclude
    n = 0
    with gzip.open(alt_gz, 'rt') as fi, open(altcontigs, 'w') as fo:
        for line in fi:
            if line.startswith('#'):
                continue
            p = line.split('\t', 4)
            if len(p) >= 4 and p[2] in ('gene', 'transcript', 'CDS') and not std.match(p[0]):
                fo.write(line)
                n += 1
    print(f'extracted {n:,} alt-contig records -> {altcontigs}')

print('inputs ready:', {'chromAlias': os.path.exists(alias_path), 'altcontigs': os.path.exists(altcontigs)})

downloading UCSC hg38 chromAlias...


extracted 114,519 alt-contig records -> genome/gencode.v48.altcontigs.gtf
inputs ready: {'chromAlias': True, 'altcontigs': True}


In [3]:
# --- 1. Primary assembly gene models (TSS = transcript start/end by strand; CDS starts) ---
gene = pd.read_parquet(f'{REF_GENOME}/gene_annotations_hg38.parquet').copy()
tx = pd.read_parquet(f'{REF_GENOME}/transcript_annotations_hg38.parquet').copy()
cds = pd.read_parquet(f'{REF_GENOME}/cds_annotations_hg38.parquet').copy()
for d in (gene, tx, cds):
    d['gene_id'] = d['gene_id'].str.split('.').str[0]

tx['tss'] = np.where(tx.strand == '+', tx.start, tx.end)
cds['cdss'] = np.where(cds.strand == '+', cds.start, cds.end)
tss = tx.groupby('gene_id')['tss'].apply(lambda s: np.array(sorted(set(s))))
cdsg = cds.groupby('gene_id')['cdss'].apply(lambda s: np.array(sorted(set(s))))

g = gene.rename(columns={'chrom': 'chromosome', 'start': 'gene_start', 'end': 'gene_end'})
g['tss'] = g['gene_id'].map(tss)
g['cds'] = g['gene_id'].map(cdsg)
# genes with no transcript -> fall back to gene start/end for TSS
mask = g['tss'].isna()
g.loc[mask, 'tss'] = g.loc[mask].apply(
    lambda r: np.array([r.gene_start if r.strand == '+' else r.gene_end]), axis=1)
g['cds'] = g['cds'].apply(lambda x: x if isinstance(x, np.ndarray) else np.array([], dtype=int))

keep_types = {'protein_coding', 'IG_V_gene', 'IG_D_gene', 'IG_J_gene', 'IG_C_gene',
              'TR_V_gene', 'TR_D_gene', 'TR_J_gene', 'TR_C_gene'}
primary = g[g.gene_type.isin(keep_types) | g.gene_id.isin(lib_ids)].copy()
print(f'primary genes_df_subset: {len(primary):,}  (library ids covered: {len(lib_ids & set(primary.gene_id)):,}/{len(lib_ids):,})')

primary genes_df_subset: 20,686  (library ids covered: 18,931/18,942)


In [4]:
# --- 2. Alt / patch / haplotype contig gene models ---
# GENCODE names alt contigs with GenBank accessions (KI270879.1); the hg38 index uses UCSC names
# (chr22_KI270879v1_alt). Remap via UCSC chromAlias.
alias = pd.read_csv(f'{GENOME}/hg38.chromAlias.txt', sep='\t')
alias.columns = [c.lstrip('# ') for c in alias.columns]
gb2ucsc = dict(zip(alias['genbank'].dropna(), alias['ucsc']))

gid_re = re.compile(r'gene_id "([^"]+)"')
gn_re = re.compile(r'gene_name "([^"]+)"')
gt_re = re.compile(r'gene_type "([^"]+)"')
rows = []
with open(f'{GENOME}/gencode.v48.altcontigs.gtf') as f:
    for line in f:
        p = line.rstrip('\n').split('\t')
        if len(p) < 9:
            continue
        m = gid_re.search(p[8])
        if not m:
            continue
        rows.append((p[0], p[2], int(p[3]), int(p[4]), p[6], m.group(1).split('.')[0],
                     (gn_re.search(p[8]).group(1) if gn_re.search(p[8]) else None),
                     (gt_re.search(p[8]).group(1) if gt_re.search(p[8]) else None)))
alt = pd.DataFrame(rows, columns=['chrom_gb', 'feat', 'start', 'end', 'strand',
                                  'gene_id', 'gene_name', 'gene_type'])
alt['chromosome'] = alt['chrom_gb'].map(gb2ucsc)
alt = alt[alt['chromosome'].notna()]

a_tx = alt[alt.feat == 'transcript'].assign(tss=lambda d: np.where(d.strand == '+', d.start, d.end))
a_cds = alt[alt.feat == 'CDS'].assign(cdss=lambda d: np.where(d.strand == '+', d.start, d.end))
a_tss = a_tx.groupby('gene_id')['tss'].apply(lambda s: np.array(sorted(set(s))))
a_cdsg = a_cds.groupby('gene_id')['cdss'].apply(lambda s: np.array(sorted(set(s))))

alt_genes = alt[alt.feat == 'gene'].rename(columns={'start': 'gene_start', 'end': 'gene_end'}).copy()
alt_genes['tss'] = alt_genes['gene_id'].map(a_tss)
alt_genes['cds'] = alt_genes['gene_id'].map(a_cdsg)
m = alt_genes['tss'].isna()
alt_genes.loc[m, 'tss'] = alt_genes.loc[m].apply(
    lambda r: np.array([r.gene_start if r.strand == '+' else r.gene_end]), axis=1)
alt_genes['cds'] = alt_genes['cds'].apply(lambda x: x if isinstance(x, np.ndarray) else np.array([], dtype=int))

# add protein-coding / IG-TR alt genes + any alt gene the library targets
alt_keep = alt_genes[alt_genes.gene_type.isin(keep_types) | alt_genes.gene_id.isin(lib_ids)].copy()
alt_keep = alt_keep[primary.columns]   # conform to primary schema
print(f'alt-contig genes kept: {len(alt_keep):,}  (protein_coding {int((alt_keep.gene_type=="protein_coding").sum()):,})')
print('library alt genes present:', sorted(alt_keep[alt_keep.gene_id.isin(lib_ids)].gene_name.unique())[:12])

alt-contig genes kept: 3,361  (protein_coding 3,166)
library alt genes present: ['CCL3L3', 'GSTT1', 'HLA-DRB3', 'HLA-DRB4', 'LILRA3']


In [5]:
# --- 3. Combine + save ---
genes_df_subset = pd.concat([primary, alt_keep], ignore_index=True)
genes_df_subset = genes_df_subset.drop_duplicates('gene_id').reset_index(drop=True)
genes_df_subset.to_parquet(f'{GENOME}/genes_df_subset_for_sgRNA_annotation.parquet')
genes_df_subset.to_csv(f'{GENOME}/genes_df_subset_for_sgRNA_annotation.csv', index=False)

print(f'genes_df_subset saved: {len(genes_df_subset):,} genes '
      f'({len(primary):,} primary + {len(alt_keep):,} alt)')
print('library ids now covered:',
      f'{len(lib_ids & set(genes_df_subset.gene_id)):,}/{len(lib_ids):,}')
genes_df_subset[genes_df_subset.gene_name.isin(['GSTT1', 'LILRA3'])][
    ['gene_id', 'gene_name', 'chromosome', 'gene_start', 'gene_end', 'strand']]

genes_df_subset saved: 24,047 genes (20,686 primary + 3,361 alt)
library ids now covered: 18,936/18,942


,gene_id,gene_name,chromosome,gene_start,gene_end,strand
22295,ENSG00000276175,LILRA3,chr19_GL949746v1_alt,268530,280828,-
22327,ENSG00000273884,LILRA3,chr19_GL949747v2_alt,268410,280719,-
22481,ENSG00000278046,LILRA3,chr19_GL949753v2_alt,268785,281093,-
22526,ENSG00000275841,LILRA3,chr19_KI270938v1_alt,270964,275840,-
22709,ENSG00000277656,GSTT1,chr22_KI270879v1_alt,270314,278855,-
